    from learndjango import models
    from django.db.models import Count, Max, Min, Avg
    # 大于
    a=models.Students.objects.filter(age__gt=18).values()

    # 小于
    b=models.Students.objects.filter(age__lt=18).values()

    # 大于等于
    c=models.Students.objects.filter(age__gte=18).values()

    # <=
    d=models.Students.objects.filter(age__lte=18).values()

    # like '%zhang%'
    e=models.Students.objects.filter(name__contains='zhang').values()

    # count(*)
    f=models.Students.objects.count()

    # max(age)
    g=models.Students.objects.aggregate(Max('age'))

    # min(age)
    h=models.Students.objects.aggregate(Min('age'))

    # avg(age)
    i=models.Students.objects.aggregate(Avg('age'))

    # in
    j=models.Students.objects.filter(id__in=[1, 2]).values()


    # ASC
    k=models.Students.objects.all().order_by('age').values()


    # DESC（字段名前加 - ）
    l=models.Students.objects.all().order_by('-age').values()
    print(a,b,c,d,f,g,h,i,j,k,l,sep='\n')

通过models.表名.objects.filter(参数=值).values()获取到数据后

有哪些方式实现只需要第一条的数据？

1. 
models.Students.objects.filter(name='zhangsan').values().first()

# QuerySet 实现了 __getitem__ 方法
# 所以可以像列表一样用索引和切片
# 所以可以用如下方式

2.取索引第一条 但如果None 则会报错 IndexError
models.Students.objects.filter(name='zhangsan').values()[0]

3. 切片
models.Students.objects.filter(name='zhangsan').values()[0:1]

4. 
models.Students.objects.filter(name='zhangsan').values()[:1]


完成的查询接口
GET 与 POST的区别
- GET一般用于从服务端获取数据，POST一般用于提交数据

- GET也可以用于提交数据，POST也可以获取数据

- GET参数是在url上传递的，POST通过body传输

- POST请求只是相对GET安全


@require_http_methods(['GET'])
def uniq_stu_info(request):

    name = request.GET.get('name')

    stu_info = models.Students.objects.filter(name=name).values()

    return JsonResponse(
        {'msg':list(stu_info)},
        status = 200
    )

# {"msg": [{"id": 1, "name": "zhangsan", "age": 18, "phone": "13024354657", "create_time": "2026-06-24T22:01:35Z"}]}

@require_http_methods(['GET'])
def get_stu_score(request):

    stu_id = request.GET.get('stu_id')

    stu_scores = models.Score.objects.filter(stu_id=stu_id).values()

    return JsonResponse(
        {'msg':list(stu_scores)},
        status = 200
    )

http://127.0.0.1:8000/stuScores/?stu_id=901

# {"msg": [{"id": 1, "stu_id": 901, "c_name": "math", "score": 99.0}, {"id": 2, "stu_id": 901, "c_name": "English", "score": 100.0}]}


@require_http_methods(['GET'])
def get_stu_age(request):

    name = request.GET.get('name')

    stu_scores = models.Students.objects.filter(name=name).values('age')

    return JsonResponse(
        {'msg':list(stu_scores)},
        status = 200
    )

http://127.0.0.1:8000/getStuAge/?name=lisi

# {"msg": [{"age": 20}]}


@require_http_methods(['GET'])
def get_course_name(request):

    id = request.GET.get('id')

    c_name = models.Course.objects.filter(id=id).values()

    return JsonResponse(
        {'msg': list(c_name)},
        status = 200
    )

http://127.0.0.1:8000/courseName/?id=1

# {"msg": [{"id": 1, "c_name": "math"}]}


@require_http_methods(['GET'])
def get_all_courses(request):

    courses = models.Course.objects.all().values()
    return JsonResponse(
        {'msg':list(courses)},
        status = 200
    )

http://127.0.0.1:8000/allCourses/?id=2

# {"msg": [{"id": 1, "c_name": "math"}, {"id": 2, "c_name": "English"}]}

@require_http_methods(['POST'])
def get_phone_num(request):
    name = request.POST.get('name')
    age = request.POST.get('age')

    phone_Num = models.Students.objects.filter(name=name,age=age).values('phone')
    
    return JsonResponse(
        {'msg':list(phone_Num)},
        status = 200
    )

{
    "msg": [
        {
            "phone": "13264553856"
        }
    ]
}

@require_http_methods(['POST'])
def get_same_age_stu(request):
    age = request.POST.get('age')

    students = models.Students.objects.filter(age=age).values('name')

    return JsonResponse(
        {'msg':list(students)},
        status = 200
    )

{
    "msg": [
        {
            "name": "lisi"
        }
    ]
}

@require_http_methods(['POST'])
def get_stu_course(request):
    stu_id = request.POST.get('stu_id')

    courses = models.Score.objects.filter(stu_id=stu_id).values('c_name')
        
    return JsonResponse(
        {'msg': list(courses)},
        status = 200
    )

{
    "msg": [
        {
            "c_name": "math"
        },
        {
            "c_name": "English"
        }
    ]
}

@require_http_methods(['POST'])
def get_avg_age(request):
    
    # age = request.POST.get('age')

    avg_age = models.Students.objects.aggregate(Avg('age'))
        
    return JsonResponse(
        {'msg': avg_age},
        status = 200
    )

{
    "msg": {
        "age__avg": 19.0
    }
}


@require_http_methods(['POST'])
def get_course_score(request):
    c_name = request.POST.get('c_name')

    scores = models.Score.objects.filter(c_name=c_name).values('c_name','score')
        
    return JsonResponse(
        {'msg': list(scores)},
        status = 200
    )

{
    "msg": [
        {
            "c_name": "math",
            "score": 99.0
        },
        {
            "c_name": "math",
            "score": 98.0
        }
    ]
}

models.py 

class Teacher(models.Model):

    t_id = models.AutoField(primary_key=True)
    # ALTER TABLE learndjango_teacher AUTO_INCREMENT = 701; 数据库跑一下，从701开始自增，Django只支持从1开始自增
    teacher_nm = models.CharField(max_length=64)
    c_name = models.CharField(max_length=64)

    def __str__(self):
        return f"{self.teacher_nm} - {self.c_name}"


-----
views.py

@require_http_methods(['POST'])
def create_teacher_info(request):

    teacher_nm = request.POST.get('teacher_nm')
    c_name = request.POST.get('c_name')

    if teacher_nm and c_name:
        teacher_obj = models.Teacher.objects.create(teacher_nm=teacher_nm,c_name=c_name)
        print(teacher_nm,c_name)
    else:
        return JsonResponse(
            {'msg':'Missing mandatory columns.'},
            status = 400
        )

    return JsonResponse(
        {'msg':f'{teacher_obj} info is created.'}
    )



总结：
create 和 save的区别
1. 新建(insert into)
create - 一步完成

teacher_obj = models.Teacher.objects.create(teacher_nm=teacher_nm,c_name=c_name)

save - 分两步
teacher_info = models.Teacher(teacher_nm=teacher_nm,c_name=c_name)
teacher_info.save()

2. 更新(update set)
create只能应用于新建
save 更新 -
teacher_info = models.Teacher.objects.get(t_id=t_id)
teacher_info.teacher_nm = 'zhaoliu'
teacher_info.save()

3. django的自增只能从1开始，如果要从某一数字开始，可以在建表后插入第一条数据前配合query ：ALTER TABLE learndjango_teacher AUTO_INCREMENT = 701;